[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natrask/AESCAPE/blob/main/notebooks/01_agentic_patterns.ipynb)

# The Agentic Framework Zoo

**AESCAPE 2026 · SC03: Introduction to Agentic Workflows**

There are a lot of agent frameworks. They are mostly not competing implementations of the
same idea — they are different answers to one question:

> **Who owns control flow?**

This notebook walks through four architectures. For each you get:

1. **the syntax** — what the real library's code looks like,
2. **a toy** — the *same* trivial task every time, so you are comparing syntax and nothing else,
3. **an exemplar** — a problem where that pattern genuinely earns its complexity.

| | Control flow | Lookahead | Roles | Reach for it when |
|---|---|---|---|---|
| **ReAct** | the model, step by step | none | one | open-ended, errors cheap |
| **LangGraph** | you, as a graph | none | many, fixed | you need guarantees |
| **AutoGen** | emergent, via chat | none | many, fluid | expertise must collide |
| **MPC** | a simulator ranks plans | *N* steps | one + world model | wrong actions are costly |

ReAct and LangGraph are run with the real libraries. AutoGen appears twice: once in the real
framework's syntax, and once in ~15 lines of plain Python, to demystify the framework.

**Contents**
- Part 0 — Setup
- Part 1 — ReAct
- Part 2 — Graphs (LangGraph)
- Part 3 — Conversational multi-agent (AutoGen)
- Part 4 — MPC-style planning
- Part 5 — **Head-to-head: ReAct vs. MPC on a real optimization problem**
- Part 6 — Where to go next

---
## Part 0 — Setup

### Getting a key

**Free, no credit card.** Get a Gemini key at <https://aistudio.google.com/apikey> and sign in
with any Google account.

> **Keep it free.** Use a Google account/project where billing has *never* been enabled in AI
> Studio. If billing is on, your key is no longer on the free tier and calls are billed per token.
> A `429 RESOURCE_EXHAUSTED` means you hit the per-minute limit (we retry automatically), used the
> daily budget, or enabled billing.

**In Colab**, put the key in the *Secrets* panel (key icon, left sidebar) under the name
`GEMINI_API_KEY`. **Locally**, `export GEMINI_API_KEY=...`

**Never paste a key into a cell.** It ends up in the `.ipynb` JSON, your shell history, and git.

In [ ]:
import sys
if 'google.colab' in sys.modules:
    %pip install -U -q "google-genai<2.13" "google-auth==2.49.0" langgraph


In [ ]:
import os, json, time, re, math, textwrap
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint, solve_ivp
from scipy.optimize import minimize_scalar

plt.rcParams['figure.dpi'] = 100

In [ ]:
try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')
if not API_KEY:
    raise RuntimeError("No GEMINI_API_KEY found. Set it in the Colab Secrets panel (key icon) "
                       "or as an environment variable.")

from google import genai
from google.genai import types as gtypes
import importlib.metadata as _md
print("google-genai", _md.version("google-genai"))
# If that prints an old 0.x/1.x version, Colab is still holding its preinstalled copy:
# Runtime -> Restart session, then run from the top.

client = genai.Client(api_key=API_KEY)

# Free-tier limits for this model, confirmed Sept 2026: 15 RPM, 250k TPM, 1000 RPD.
# Agent loops are bursty, so the per-minute cap is what the retry wrapper absorbs;
# the daily cap is the one that ends a session.
MODEL = "gemini-3.1-flash-lite"
print(f"model = {MODEL}")

### One call, with retries

Free-tier limits are per-minute, and agent loops fire in bursts. We back off and retry rather
than crashing halfway through a demo.

In [ ]:
def generate_with_retry(*, contents, config=None, max_attempts=6):
    """client.models.generate_content with exponential backoff on 429."""
    delay = 4.0
    for attempt in range(max_attempts):
        try:
            return client.models.generate_content(
                model=MODEL, contents=contents, config=config)
        except Exception as e:
            msg = str(e)
            retryable = ("429" in msg or "RESOURCE_EXHAUSTED" in msg
                         or "quota" in msg.lower() or "503" in msg)
            if not retryable or attempt == max_attempts - 1:
                raise
            print(f"[rate limit] attempt {attempt+1}: sleeping {delay:.1f}s...")
            time.sleep(delay)
            delay = min(delay * 2, 60.0)


def llm_text(system_prompt, user_prompt, temperature=0.2):
    """Plain text in, plain text out. No tools. The simplest possible call."""
    resp = generate_with_retry(
        contents=user_prompt,
        config=gtypes.GenerateContentConfig(
            system_instruction=system_prompt, temperature=temperature,
            automatic_function_calling=gtypes.AutomaticFunctionCallingConfig(disable=True)))
    return resp.text or ""

In [ ]:
# Smoke test. If this prints a sentence, you are ready.
print(llm_text("You are terse.", "Say hello in one short sentence."))

### Swapping backends

Everything below goes through `llm_text` and `run_agent`. To use Anthropic or OpenAI instead,
you replace the body of those two functions and nothing else — the loops, graphs, and
comparisons are provider-agnostic.

The three things that differ between providers:

| | Gemini | Anthropic | OpenAI |
|---|---|---|---|
| tool schema | `FunctionDeclaration(...)` | `{"name", "description", "input_schema"}` | `{"type":"function","function":{...}}` |
| model asks | `part.function_call` | `content` block, `type="tool_use"` | `message.tool_calls[]` |
| you answer | `Part.from_function_response` | `{"type":"tool_result","tool_use_id":...}` | `{"role":"tool","tool_call_id":...}` |

Same three ideas, three spellings. That similarity is exactly why a thin shim works.

```python
# Anthropic, for reference — model IDs as of Sept 2026:
#   claude-opus-5 · claude-sonnet-5 · claude-haiku-4-5
import anthropic
client = anthropic.Anthropic(api_key=...)
resp = client.messages.create(model="claude-sonnet-5", max_tokens=2048,
                              system=system_prompt, tools=tool_schemas,
                              messages=messages)
```

---
# Part 1 — ReAct

> **Control flow: the model.** At each step it looks at everything so far and picks one action.

ReAct (Reason + Act, [Yao et al. 2022](https://arxiv.org/abs/2210.03629)) is the baseline every
other pattern is a reaction to. There is no framework — it is a `for` loop and a dispatch table.

### 1.1 A tool is a function plus a schema

Two pieces, both written by you: an ordinary Python function, and a JSON-schema description of
its signature that you hand to the model.

The schema is not bookkeeping — it is prompt engineering. `name` and `description` decide whether
the model picks *this* tool over the others, and every value you forbid with `enum` is a failure
mode you never have to debug.

In [ ]:
def calculator(op: str, a: float, b: float) -> dict:
    """Perform one arithmetic operation."""
    ops = {"add": a + b, "sub": a - b, "mul": a * b,
           "div": a / b if b != 0 else float('nan')}
    if op not in ops:
        return {"error": f"unknown op {op!r}; valid: add, sub, mul, div"}
    return {"result": ops[op]}


CALC_PARAMS = {
    "type": "object",
    "properties": {
        "op": {"type": "string", "enum": ["add", "sub", "mul", "div"],
               "description": "operation to perform"},
        "a":  {"type": "number"},
        "b":  {"type": "number"},
    },
    "required": ["op", "a", "b"],
}

def calc_schema():
    return gtypes.FunctionDeclaration(
        name="calculator",
        description="Perform one arithmetic operation.",
        parameters=CALC_PARAMS)

print(json.dumps(CALC_PARAMS, indent=2))

### 1.2 The loop

Three moving parts:

1. Call the model with the running conversation and the tool schemas.
2. If the response contains a `function_call`, run the tool and append its output as an observation.
3. If the response is plain text, that is the final answer.

Tool dispatch is wrapped in `try/except` so a malformed call comes back to the model as an
observation instead of killing the run. **You will see this trigger.** An agent recovering from
its own bad tool call is the system working.

In [ ]:
def run_agent(system_prompt, user_prompt, tool_schemas, tool_fns,
              max_steps=12, temperature=0.2, verbose=True):
    """Minimal ReAct loop. Returns (final_text, transcript)."""
    contents = [gtypes.Content(role="user",
                               parts=[gtypes.Part.from_text(text=user_prompt)])]
    cfg = gtypes.GenerateContentConfig(
        system_instruction=system_prompt,
        tools=[gtypes.Tool(function_declarations=tool_schemas)],
        temperature=temperature,
        automatic_function_calling=gtypes.AutomaticFunctionCallingConfig(disable=True))

    transcript = []
    for step in range(max_steps):
        resp = generate_with_retry(contents=contents, config=cfg)
        parts = resp.candidates[0].content.parts or []
        contents.append(resp.candidates[0].content)

        calls = [p.function_call for p in parts if getattr(p, "function_call", None)]
        if not calls:
            text = "".join(getattr(p, "text", "") or "" for p in parts)
            transcript.append(("final", text))
            if verbose: print(f"[{step}] FINAL: {text[:160]}")
            return text, transcript

        obs = []
        for fc in calls:
            name, args = fc.name, dict(fc.args or {})
            if verbose: print(f"[{step}] CALL {name}({args})")
            try:
                result = tool_fns[name](**args)
            except Exception as e:
                result = {"error": f"{type(e).__name__}: {e}"}   # errors are observations
            transcript.append((name, args, result))
            if verbose: print(f"[{step}]   -> {json.dumps(result)[:160]}")
            obs.append(gtypes.Part.from_function_response(name=name, response=result))
        contents.append(gtypes.Content(role="user", parts=obs))

    transcript.append(("final", "(max_steps reached)"))
    return "(max_steps reached)", transcript

### 1.3 The toy task

Every pattern in this notebook gets the *same* trivial job, so you are comparing syntax rather
than problems: **compute `13 * 47 + 8` using the calculator tool.** The answer is 619.

The toy is deliberately too easy to justify any of these architectures. That is the point — with
the problem out of the way you can see the raw shape of each framework.

In [ ]:
TOY_SYSTEM = ("You are a careful arithmetic assistant. Use the calculator tool for every "
              "computation; never do arithmetic in your head.")
TOY_USER = "Compute 13 * 47 + 8. Return just the final number."

answer, transcript = run_agent(TOY_SYSTEM, TOY_USER,
                               tool_schemas=[calc_schema()],
                               tool_fns={"calculator": calculator},
                               max_steps=8)
print("\n=== Final answer:", answer)

### 1.4 When to reach for ReAct

**Use it when** the procedure cannot be specified in advance, mistakes are cheap and recoverable,
and the tool surface is small.

**Its exemplar** is adaptive mesh refinement (notebook `02_agent_hackathon.ipynb`): the number of
refinement cycles depends on error you have not measured yet, a bad refinement just produces a
mesh you discard, and the model must react to observations it could not have predicted. Greedy,
one-step-at-a-time decision making is not a compromise there — it is the correct algorithm.

**Where it starts to hurt** is everything in Parts 2–4.

---
# Part 2 — Graphs (LangGraph)

> **Control flow: you, in code.** The model fills in nodes; the graph decides what runs next.

The pressure that produces this pattern: you need a **guarantee**. Something must *always*
happen — verification, logging, a human sign-off — and "the model usually remembers to" is not
good enough.

### 2.1 The syntax

You declare a typed state, nodes that transform it, and edges. A node returns a *partial* update
to the state; the graph merges it. The router is ordinary Python, which means the control flow is
testable, diffable, and reviewable.

### 2.2 The toy, as a graph

Same arithmetic task — but now an **independent check** runs before any answer is returned, and
the retry budget is enforced in Python rather than requested in English.

Notice what the model is *not* allowed to do: it cannot skip `verify`, and it cannot talk its way
into a fourth attempt.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class GraphState(TypedDict):
    value: float
    attempts: int
    solves: int
    verified: bool

def n_compute(s):
    """The 'agent' node: ask the model and parse a number out of the reply."""
    txt = llm_text("Reply with only a number, no words, no units.",
                   f"What is 13 * 47 + {8 + s['attempts']}? Reply with only the number.")
    m = re.search(r"-?\d+(?:\.\d+)?", txt)
    val = float(m.group()) if m else float('nan')
    return {"value": val, "solves": s["solves"] + 1}

def n_verify(s):
    """Independent check. Recomputes rather than trusting the claim."""
    return {"verified": bool(abs(s["value"] - (13 * 47 + 8)) < 1e-9)}

def n_repair(s):
    return {"attempts": s["attempts"] + 1}

MAX_ATTEMPTS = 3
def route(s):
    if s["verified"]:                 return END
    if s["attempts"] >= MAX_ATTEMPTS: return END      # bounded in code, not in the prompt
    return "repair"

def build_graph(compute_fn):
    g = StateGraph(GraphState)
    g.add_node("compute", compute_fn)
    g.add_node("verify", n_verify)
    g.add_node("repair", n_repair)
    g.set_entry_point("compute")
    g.add_edge("compute", "verify")
    g.add_conditional_edges("verify", route, {"repair": "repair", END: END})
    g.add_edge("repair", "compute")
    return g.compile()

def run_graph(app, state):
    """Stream node-by-node so we can print the path the graph actually took."""
    path = []
    for step in app.stream(state, stream_mode="updates"):
        for node, update in step.items():
            path.append(node)
            state = {**state, **update}
    return state, path

app = build_graph(n_compute)
final, path = run_graph(app, {"value": 0.0, "attempts": 0, "solves": 0, "verified": False})
print("path:", " -> ".join(path))
print(f"value={final['value']}  verified={final['verified']}  "
      f"attempts={final['attempts']}  solves={final['solves']}")

### 2.3 The guarantee, demonstrated

The point of a graph is what happens when things go *wrong*. Below, the compute node never
produces the right answer. The run still terminates, still bounded, and still reports honestly
that it failed — with no prompt asking it to.

In [ ]:
def n_stuck(s):
    return {"value": 42.0, "solves": s["solves"] + 1}

bad, bpath = run_graph(build_graph(n_stuck),
                       {"value": 0.0, "attempts": 0, "solves": 0, "verified": False})
print("path:", " -> ".join(bpath))
print(f"verified={bad['verified']}  attempts={bad['attempts']}  solves={bad['solves']}")
assert bad["attempts"] == MAX_ATTEMPTS and not bad["verified"]
print("\nBounded, verified-or-honest, and reproducible. No prompt could have promised that.")

### 2.4 When to reach for a graph

**Use it when** the control flow is a *requirement*, not a choice:

| Requirement | Who enforces it |
|---|---|
| Every reported result is verified | the graph, not the prompt |
| At most *N* attempts | the router, in Python |
| A failed check triggers repair, not a retry | a conditional edge |
| The run is reproducible from a checkpoint | typed state |

**Its exemplar** is the production FEM workflow: a result that goes into a report has to have
been verified, every time, whatever the model felt like doing that morning.

**The cost:** you now maintain a graph. If your control flow genuinely is "whatever seems next,"
a graph is ceremony — use ReAct.

---
# Part 3 — Conversational multi-agent (AutoGen)

> **Control flow: emergent, from the conversation.** Who speaks next determines what happens next.

The pressure that produces this pattern: genuinely **different expertise** needs different system
prompts, different tools, and different incentives — and you want the disagreement on the record.

### 3.1 The syntax

```python
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination

modeler = AssistantAgent("modeler", model_client=client,
    system_message="You propose discretizations. Be concrete and specific.")

analyst = AssistantAgent("analyst", model_client=client,
    system_message="""You are a numerical analyst. Challenge the modeler's proposal on
    stability, conditioning, and convergence rate. Do not be agreeable; your value is
    in the objection.""")

critic = AssistantAgent("critic", model_client=client,
    system_message="Judge the exchange. Say APPROVE only when genuinely convinced.")

team = RoundRobinGroupChat([modeler, analyst, critic],
    termination_condition=TextMentionTermination("APPROVE"))

await team.run(task="Choose a discretization for advection-dominated flow.")
```

Note `autogen_agentchat` (v0.4+), not the older `autogen`/`pyautogen` packages — the API changed
substantially and most tutorials you will find online are for the old one.

In [ ]:
try:
    import autogen_agentchat
    HAVE_AUTOGEN = True
    print("autogen_agentchat is available.")
except ImportError:
    HAVE_AUTOGEN = False
    print("autogen_agentchat not installed — using the from-scratch version below.")

### 3.2 The same thing in 15 lines

A round-robin group chat is a list of system prompts, a shared transcript, and a stopping rule.

In [ ]:
def group_chat(agents, task, max_rounds=3, terminate_on="APPROVE", verbose=True):
    """agents: list of (name, system_prompt). Returns the transcript."""
    transcript = [("user", task)]
    for rnd in range(max_rounds):
        for name, system in agents:
            history = "\n\n".join(f"[{who}] {what}" for who, what in transcript)
            reply = llm_text(system, f"Conversation so far:\n\n{history}\n\nYour turn, {name}.")
            reply = reply.strip()
            transcript.append((name, reply))
            if verbose:
                print(f"\n--- [{name}] " + "-" * 50)
                print(textwrap.fill(reply, 92)[:900])
            if terminate_on in reply:
                return transcript
    return transcript

### 3.3 The exemplar: a design review

The toy task does not work here, and that is informative — there is nothing to argue about in
`13 * 47 + 8`. A conversation pattern needs a question with genuine tension in it.

So: **choosing a discretization for advection-dominated flow.** No single right answer, real
tradeoffs, and the *reasoning* is the deliverable.

In [ ]:
AGENTS = [
    ("modeler",
     "You propose discretizations for PDE problems. Be concrete and specific: name the method "
     "and the parameters. Two or three sentences. Respond to objections rather than repeating "
     "yourself."),
    ("analyst",
     "You are a numerical analyst. Challenge the modeler's proposal on stability, conditioning, "
     "and convergence rate. Be specific about the failure mode you are worried about. Do not be "
     "agreeable -- your value to this conversation is the objection. Two or three sentences."),
    ("critic",
     "You judge the exchange between a modeler and a numerical analyst. If the analyst's "
     "objection has been genuinely answered, reply with exactly APPROVE followed by one sentence "
     "of justification. Otherwise state in one sentence what is still unresolved. Do not say "
     "APPROVE merely because the discussion is polite or has gone on a while."),
]

TASK = ("Choose a spatial discretization for steady advection-diffusion at Peclet number ~500 "
        "on an unstructured triangular mesh. State the method and any stabilization parameter.")

tr = group_chat(AGENTS, TASK, max_rounds=2)
print(f"\n\n=== {len(tr)-1} replies; terminated: {'APPROVE' in tr[-1][1]}")

### 3.4 When to reach for conversation — and the honest caveat

**Use it when** disagreement is the product: the roles have different tools and different
incentives, and you want an auditable argument rather than a confident paragraph.

**The caveat, stated plainly:** this is the easiest of the four patterns to fool yourself with.

- Three LLMs agreeing is **not** three experts agreeing. They share a prior, a training corpus,
  and a tendency toward agreeableness. A "critic" that approves everything has told you nothing.
- Distinct **tools** per role helps far more than distinct adjectives in the system prompt. An
  analyst who can actually *run* a stability calculation is a different thing from one instructed
  to "be skeptical."
- Watch the run above for the critic approving too early. If it does, that is the pattern's
  characteristic failure — not a bug in the prompt.

**Cheapest useful version:** two agents, one of which has a tool the other lacks.

---
# Part 4 — MPC-style planning

> **Control flow: a simulator.** The LLM proposes candidate plans; a forward model ranks them.

Model Predictive Control, borrowed from process control: at each step, look ahead, plan the next
*N* actions against a world model, **execute only the first**, then re-plan from the new state.

The agentic version swaps one box — the LLM proposes the candidates, and the simulator (not the
LLM) decides which is best:

```
state ──▶ plan (LLM, N candidates) ──▶ simulate (forward model) ──▶ act (first step only)
  ▲                                                                          │
  └──────────────────────── re-plan from the new state ◀────────────────────┘
```

The LLM contributes what it is good at: proposing plausible candidates from context.
The simulator contributes what it is good at: **being right**.

### 4.1 The environment

A projectile with quadratic drag. The agent will only ever see `evaluate(angle) -> range`; it
does not get to look at the ODE.

$$\ddot{x} = -c_d |v|\dot{x}, \qquad \ddot{y} = -g - c_d|v|\dot{y}$$

with $v_0 = 50$ m/s, $c_d = 0.01$ m$^{-1}$, $g = 9.81$ m/s².

Without drag the optimum is exactly 45°. With drag it shifts lower — so the model's prior is
*almost* right, which is the interesting case.

In [ ]:
V0, CD, G = 50.0, 0.01, 9.81

def _rhs_drag(s, t):
    x, y, vx, vy = s
    v = np.sqrt(vx*vx + vy*vy)
    return [vx, vy, -CD*v*vx, -G - CD*v*vy]

def simulate_range(angle_deg):
    """Horizontal range (m) for a given launch angle. This is the expensive action."""
    a = np.deg2rad(angle_deg)
    s0 = [0.0, 0.0, V0*np.cos(a), V0*np.sin(a)]
    t = np.linspace(0, 20, 4001)
    sol = odeint(_rhs_drag, s0, t)
    y = sol[:, 1]
    hit = np.where((y[:-1] >= 0) & (y[1:] < 0))[0]
    if len(hit) == 0:
        return float('nan')
    i = hit[0]
    frac = y[i] / (y[i] - y[i+1])
    return float(sol[i, 0] + frac * (sol[i+1, 0] - sol[i, 0]))

# Ground truth, for scoring only. No agent gets to see this.
_res = minimize_scalar(lambda a: -simulate_range(a), bounds=(1, 89),
                       method='bounded', options={'xatol': 1e-4})
TRUE_OPT, TRUE_RANGE = float(_res.x), float(-_res.fun)
print(f"True optimum: {TRUE_OPT:.4f} deg  ->  {TRUE_RANGE:.4f} m")
print(f"(scipy.minimize_scalar needed {_res.nfev} evaluations to find it)")

angles = np.linspace(5, 85, 81)
ranges = [simulate_range(a) for a in angles]
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(angles, ranges, 'b-')
ax.axvline(TRUE_OPT, color='k', ls='--', lw=1, label=f'optimum {TRUE_OPT:.2f} deg')
ax.axvline(45, color='r', ls=':', lw=1, label='45 deg (no-drag prior)')
ax.set_xlabel("launch angle (deg)"); ax.set_ylabel("range (m)")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

### 4.2 First, why the naive version is degenerate

An honest detour, because it is the mistake everyone makes on their first MPC agent.

MPC needs a forward model that is **cheaper than acting**. If you "simulate" a candidate angle by
calling `simulate_range` on it, you have already paid the full cost of the action — so
lookahead buys you nothing. You have built an expensive way to do a grid search.

**The fix:** the forward model has to be a genuine *surrogate*. Here, a quadratic least-squares
fit through the best few observations — essentially free, and accurate near the peak, which is
the only place accuracy matters.

In [ ]:
def fit_surrogate(obs, k=5):
    """Quadratic LSQ fit through the k best observations. This is the cheap forward model."""
    if len(obs) < 3:
        return None
    best = sorted(obs, key=lambda o: -o[1])[:max(k, 3)]
    a = np.array([o[0] for o in best], float)
    r = np.array([o[1] for o in best], float)
    if len(np.unique(a)) < 3:
        return None
    return np.polyfit(a, r, 2)

def surrogate_predict(coef, angle):
    return float(np.polyval(coef, angle))

def surrogate_vertex(coef):
    """Argmax of the fitted quadratic, if it opens downward."""
    A, B, _ = coef
    return None if A >= 0 else float(-B / (2 * A))

# Sanity check: three points is enough to locate the peak roughly.
demo = [(a, simulate_range(a)) for a in (20.0, 45.0, 70.0)]
c = fit_surrogate(demo)
print(f"3 observations -> surrogate vertex at {surrogate_vertex(c):.2f} deg "
      f"(true {TRUE_OPT:.2f} deg)")

### 4.3 The MPC loop

Read the loop for what the LLM is **not** allowed to do: it never decides which plan is best.
It generates hypotheses; the forward model adjudicates. That division is the entire pattern.

In [ ]:
def mpc_optimize(propose, evaluate, budget=12, n_candidates=6,
                 seeds=(20.0, 45.0, 70.0), verbose=True):
    """Plan -> simulate (surrogate) -> act (ONE real evaluation) -> re-plan."""
    obs = [(a, evaluate(a)) for a in seeds]
    while len(obs) < budget:
        coef = fit_surrogate(obs)                                  # 1. world model
        cands = [c for c in propose(obs, n_candidates, coef) if 0 < c < 90]   # 2. plan
        if not cands:
            print("  proposer returned no candidates in (0, 90); stopping early")
            break
        if coef is None:
            best = cands[0]
        else:
            best = max(cands, key=lambda a: surrogate_predict(coef, a))       # 3. simulate
        obs.append((best, evaluate(best)))                         # 4. act: one action only
        if verbose:
            print(f"  n={len(obs):2d}  proposed {len(cands)}  acted on {best:6.2f}"
                  f"  -> {obs[-1][1]:7.3f}")
    return obs

In [ ]:
def heuristic_proposer(obs, n, coef):
    """A non-LLM proposer: bracket the incumbent, plus the surrogate's vertex.
    Part 5 swaps in an LLM here; the loop itself does not change."""
    best_angle = max(obs, key=lambda o: o[1])[0]
    spread = max(1.0, 20.0 / (1 + len(obs)))
    cands = list(np.linspace(best_angle - spread, best_angle + spread, n - 1))
    v = surrogate_vertex(coef) if coef is not None else None
    cands.append(v if v is not None else best_angle)
    return cands

evals = {"n": 0}
def counted_eval(a):
    evals["n"] += 1
    return simulate_range(a)

print("MPC with the heuristic proposer (no LLM):")
obs_mpc = mpc_optimize(heuristic_proposer, counted_eval, budget=12)
a_mpc, r_mpc = max(obs_mpc, key=lambda o: o[1])
print(f"\nbest = {a_mpc:.4f} deg (error {abs(a_mpc-TRUE_OPT):.4f}), "
      f"range = {r_mpc:.4f} m, evaluations = {evals['n']}")

### 4.4 When to reach for MPC

**Use it when:**
- a wrong action is **expensive or irreversible** — burning budget, committing a mesh, running an experiment;
- you have a **cheap forward model you trust more than the LLM**;
- actions compose, so looking *N* steps ahead genuinely differs from looking one step ahead.

**Don't bother when:**
- simulating a plan costs about what executing it costs — then just execute and observe (§4.2);
- you have **no** forward model, in which case MPC degenerates into the LLM grading its own
  homework: slower than ReAct and equally wrong;
- the problem is small and convex. Use `scipy.optimize` and go home.

That last one is not a joke, and Part 5 measures it.

---
# Part 5 — Head-to-head: ReAct vs. MPC

Everything so far has been assertion. This part measures.

**The experiment.** Same problem (find the optimal launch angle), same evaluation budget (12
calls to `evaluate`), two architectures:

- **ReAct** — the model picks the next angle to try, one at a time, from the history.
- **MPC** — the model proposes 6 candidates; the *surrogate* picks which one to spend the
  evaluation on.

Plus two non-LLM baselines that keep everyone honest: a uniform grid at the same budget, and
`scipy.optimize.minimize_scalar`.

**Scoring:** absolute error in the located angle, at a fixed evaluation budget.

### 5.1 The ReAct optimizer

Three read-only tools and a budget. The model chooses its own search strategy.

In [ ]:
EVAL_LOG = []

def tool_evaluate(angle_deg: float):
    """Run the black-box projectile simulator and return the horizontal range."""
    if not (0 < angle_deg < 90):
        return {"error": f"angle must be in (0, 90) degrees; got {angle_deg}"}
    if len(EVAL_LOG) >= 12:
        return {"error": "evaluation budget exhausted; call propose_answer now"}
    r = simulate_range(angle_deg)
    EVAL_LOG.append({"angle_deg": float(angle_deg), "range": r})
    return {"angle_deg": angle_deg, "range": r, "evaluations_used": len(EVAL_LOG)}

def tool_history():
    """Return all previous (angle, range) evaluations."""
    return {"evaluations": EVAL_LOG, "count": len(EVAL_LOG)}

def tool_propose_answer(angle_deg: float, rationale: str = ""):
    """State your best estimate of the optimal launch angle and stop."""
    return {"proposed_optimum_deg": angle_deg, "rationale": rationale}

OPT_TOOL_FNS = {"evaluate": tool_evaluate, "history": tool_history,
                "propose_answer": tool_propose_answer}

def opt_schemas():
    return [
        gtypes.FunctionDeclaration(
            name="evaluate",
            description="Run the black-box projectile simulator for a launch angle in (0,90) "
                        "degrees and return the horizontal range in meters.",
            parameters={"type": "object",
                        "properties": {"angle_deg": {"type": "number"}},
                        "required": ["angle_deg"]}),
        gtypes.FunctionDeclaration(
            name="history",
            description="Return all previous (angle, range) evaluations.",
            parameters={"type": "object", "properties": {}, "required": []}),
        gtypes.FunctionDeclaration(
            name="propose_answer",
            description="State your best estimate of the optimal launch angle and stop.",
            parameters={"type": "object",
                        "properties": {"angle_deg": {"type": "number"},
                                       "rationale": {"type": "string"}},
                        "required": ["angle_deg"]}),
    ]

In [ ]:
REACT_SYSTEM = """You optimize a black-box function. It takes a launch angle in degrees,
strictly between 0 and 90, and returns a horizontal range in meters. Your goal is to locate the
angle that maximizes range, as precisely as you can.

Budget: at most 12 evaluate() calls. Spend them well -- coarse first, then refine around the
best region you have found. Use history() if you lose track. When your budget is spent or you
are confident, call propose_answer with your best estimate."""

react_angle = None
EVAL_LOG.clear()
ans, tr_react = run_agent(REACT_SYSTEM, "Find the optimal angle. Budget: 12 evaluations.",
                          tool_schemas=opt_schemas(), tool_fns=OPT_TOOL_FNS,
                          max_steps=20, verbose=True)
for e in reversed(tr_react):
    if e[0] == "propose_answer":
        react_angle = float(e[1]["angle_deg"]); break
if react_angle is None and EVAL_LOG:
    react_angle = max(EVAL_LOG, key=lambda d: d["range"])["angle_deg"]
REACT_LOG = list(EVAL_LOG)
print(f"\nReAct proposed {react_angle:.4f} deg using {len(REACT_LOG)} evaluations")

### 5.2 The MPC optimizer

Same budget. The model's *only* job is to propose candidates; the surrogate decides where the
evaluation goes.

In [ ]:
def parse_numbers(text, lo=0.0, hi=90.0):
    """Pull floats out of a model reply. Robust to prose, markdown fences, and stray commas."""
    out = []
    for m in re.finditer(r"-?\d+(?:\.\d+)?", text or ""):
        v = float(m.group())
        if lo < v < hi:
            out.append(v)
    return out

def llm_proposer(obs, n, coef):
    """The LLM proposes candidates. It does NOT get to pick the winner."""
    hist = ", ".join(f"({a:.3f} -> {r:.3f})" for a, r in sorted(obs)[-12:])
    txt = llm_text(
        "You propose candidate parameter values for an optimizer. "
        "Reply with ONLY a JSON list of numbers. No prose, no explanation.",
        f"Observed (launch_angle_deg -> range_m): {hist}\n\n"
        f"Propose {n} new candidate angles strictly between 0 and 90 that are most worth "
        f"testing next to maximize range. Favour the neighbourhood of the best observations, "
        f"but include at least one exploratory value. Reply with only a JSON list.",
        temperature=0.7)
    return parse_numbers(txt)[:n]

In [ ]:
mpc_evals = {"n": 0}
def mpc_counted_eval(a):
    mpc_evals["n"] += 1
    return simulate_range(a)

print("MPC run:")
obs_head = mpc_optimize(llm_proposer, mpc_counted_eval, budget=12, n_candidates=6)
mpc_angle, mpc_range = max(obs_head, key=lambda o: o[1])
print(f"\nMPC located {mpc_angle:.4f} deg using {mpc_evals['n']} evaluations")

### 5.3 The scoreboard

In [ ]:
# Baseline 1: uniform grid at the same budget.
grid = np.linspace(5, 85, 12)
grid_obs = [(a, simulate_range(a)) for a in grid]
grid_angle = max(grid_obs, key=lambda o: o[1])[0]

# Baseline 2: scipy, unrestricted budget.
scipy_angle, scipy_evals = TRUE_OPT, _res.nfev

rows = [("uniform grid",            grid_angle,  12),
        ("MPC (surrogate-ranked)",  mpc_angle,   mpc_evals["n"]),
        ("scipy.minimize_scalar",   scipy_angle, scipy_evals)]
if react_angle is not None:
    rows.insert(1, ("ReAct (model-chosen)", react_angle, len(REACT_LOG)))

print(f"{'method':<26} {'angle (deg)':>12} {'|error|':>10} {'evals':>7}")
print("-" * 58)
for name, ang, ne in rows:
    print(f"{name:<26} {ang:>12.4f} {abs(ang-TRUE_OPT):>10.4f} {ne:>7d}")
print("-" * 58)
print(f"{'truth':<26} {TRUE_OPT:>12.4f} {0.0:>10.4f} {'--':>7}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(angles, ranges, 'b-', alpha=0.5, lw=1, label='range(angle)')
ax.scatter([a for a, _ in grid_obs], [r for _, r in grid_obs],
           marker='x', s=45, c='gray', label='uniform grid (12)')
if REACT_LOG:
    ax.scatter([d["angle_deg"] for d in REACT_LOG], [d["range"] for d in REACT_LOG],
               marker='o', s=45, c='tab:orange', alpha=0.8,
               label=f'ReAct ({len(REACT_LOG)})')
ax.scatter([a for a, _ in obs_head], [r for _, r in obs_head],
           marker='s', s=45, c='tab:green', alpha=0.8,
           label=f'MPC ({mpc_evals["n"]})')
ax.axvline(TRUE_OPT, color='k', ls='--', lw=1, label=f'truth {TRUE_OPT:.2f} deg')
ax.set_xlabel("launch angle (deg)"); ax.set_ylabel("range (m)")
ax.set_title("Where each method spent its evaluations")
ax.set_ylim(min(ranges) - 2, max(ranges) + 2)
ax.legend(loc='lower center', fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 5.4 Reading the result honestly

Look at *where the points are*, not just the final numbers. The grid spreads its budget evenly
over a region it already knows is uninteresting. MPC clusters almost everything within a few
degrees of the peak, because after three evaluations the surrogate already knows roughly where
the peak is.

**What this experiment does and does not show.**

- It **does** show that a cheap forward model converts a fixed budget into far more precision
  than either uniform sampling or step-by-step model judgment. That is the MPC thesis, and it
  holds here.
- It **does not** show that you should use an LLM for this problem. `scipy.minimize_scalar`
  finds the same answer with no model, no key, and no prompt. **If an agent cannot beat scipy on
  a 1-D smooth unimodal problem, that is worth knowing and worth saying out loud.**
- The LLM's contribution in the MPC run is *proposal diversity*, not decision quality. Swap
  `llm_proposer` for `heuristic_proposer` and compare — on this problem the difference is small,
  which tells you the surrogate is doing the work.

**So when is the LLM earning its place?** When the proposal step needs context that is not in the
data: physical priors, unit awareness, a plausible range for a parameter it has never seen,
constraints stated in prose. This projectile problem is small enough to see through — which is
exactly why it is a good place to build the intuition before you point the same machinery at a
problem where you cannot.

**Run it a few times.** The ReAct number will move around noticeably; MPC much less, because the
surrogate stabilizes it. Variance across runs *is* a result — a pattern that only works
sometimes has not been shown to work.

---
# Part 6 — Where to go next

### Stretch goals

1. **Swap the backend.** Point `llm_text` and `run_agent` at Anthropic or OpenAI. How much of the
   notebook actually changes? (Less than you would expect — that is the point of the shim table
   in Part 0.)
2. **Break a tool on purpose.** Add a sign error to `_rhs_drag`. Which method notices? Does *any*
   of them? This is the Part 2 argument in miniature.
3. **Give MPC a worse surrogate.** Replace the quadratic with a linear fit. Watch the pattern
   degrade toward greedy search — and note that nothing errors, it just quietly gets worse.
4. **Put the verifier on a graph edge.** Take the MPC result, add a `verify` node that
   re-integrates with `solve_ivp(rtol=1e-10)`, and route a FAIL back to more evaluations.
5. **Make the critic actually critical.** In Part 3, give the analyst a tool the modeler lacks
   (a Peclet-number calculator) and see whether the conversation gets sharper.

### The companion notebook

`02_agent_hackathon.ipynb` takes the ReAct pattern to a real numerical problem: agent-driven
adaptive mesh refinement for the L-shaped Poisson problem in scikit-fem, with a
manufactured-solution verifier.

### Reading

**Patterns** — [ReAct](https://arxiv.org/abs/2210.03629) ·
[LangGraph](https://langchain-ai.github.io/langgraph/) ·
[AutoGen](https://microsoft.github.io/autogen/) ·
[smolagents](https://huggingface.co/docs/smolagents)

**Agents doing science** — [FunSearch](https://www.nature.com/articles/s41586-023-06924-6)
(*Nature* 625, 2024) ·
[AlphaEvolve](https://deepmind.google/discover/blog/alphaevolve-a-gemini-powered-coding-agent-for-designing-advanced-algorithms/) ·
[ChemCrow](https://arxiv.org/abs/2304.05376) ·
[Coscientist](https://www.nature.com/articles/s41586-023-06792-0) ·
[Sakana AI Scientist](https://sakana.ai/ai-scientist/)

**Numerics** — [scikit-fem](https://scikit-fem.readthedocs.io/) ·
Dörfler, *SIAM J. Numer. Anal.* 33(3), 1996

---

### The one-slide summary

Choosing a pattern is choosing **who owns control flow**: the model (ReAct), you (LangGraph),
the conversation (AutoGen), or a simulator (MPC).

Start at the top of that list. Every step down is more machinery, more failure surface, and more
code you own. Earn each one.